# Teaching a 7B Model to Beat GPT-4o at Chess

In this notebook, we'll train **Qwen 2.5 7B Instruct** to play chess using:
1. **Supervised Fine-Tuning (SFT)** — teach the model chess move format and basic strategy
2. **GRPO (Group Relative Policy Optimization)** — reinforcement learning with Stockfish as the reward signal

All heavy compute runs on UVA's Rivanna HPC cluster via the [`rv` CLI](https://rivanna.dev). This notebook runs locally and dispatches jobs.

**Prerequisites:**
- `rv` CLI installed and configured (`rv init`)
- VPN connected to UVA Anywhere
- `.env` file with `OPENAI_API_KEY`, `HF_TOKEN`, `WANDB_API_KEY`

## 1. Setup & Prerequisites

In [1]:
# Install local dependencies (python-chess for interactive cells)
!uv pip install python-chess openai

Audited 2 packages in 5ms


In [2]:
# Check rv is installed and connected
!rv status

⠋ Fetching cluster status...⠙ Fetching cluster status...⠹ Fetching cluster status...⠸ Fetching cluster status...⠼ Fetching cluster status...⠴ Fetching cluster status...⠦ Fetching cluster status...⠧ Fetching cluster status...⠇ Fetching cluster status...⠏ Fetching cluster status...⠋ Fetching cluster status...⠙ Fetching cluster status...⠹ Fetching cluster status...⠸ Fetching cluster status...⠼ Fetching cluster status...⠴ Fetching cluster status...⠦ Fetching cluster status...⠧ Fetching cluster status...⠇ Fetching cluster status...⠏ Fetching cluster status...⠋ Fetching cluster status...⠙ Fetching cluster status...⠹ Fetching cluster status...⠸ Fetching cluster status...
Connection    OK (uva-hpc as abs6bd)
Account       meng-lab | 8,668,903 SUs remaining

Storage
  Type               Mount                   Usage       Used
  Home Directory     /home/abs6bd            28/215 GB   13%
  Scratch            /scratch/abs6bd         3.0/11 TB   27%
  Research Standard  /standard/llm-research  20.

In [3]:
# Push API keys to Rivanna (these are injected into batch jobs via rv run)
# ensure that you've copied the .env file and made it yourself; run `cp .env.example .env` and fill in your keys first!!
!rv env import .env


Imported 3 variables from .env:
  OPENAI_API_KEY=sk-p****
  HF_TOKEN=hf_U****
  WANDB_API_KEY=wand****



In [ ]:
# Download Stockfish chess engine to Rivanna
!rv run --mig bash scripts/setup_stockfish.sh

In [ ]:
# Validate that dependencies install correctly on the cluster
!rv run --mig python -c "import trl, chess, wandb, peft, datasets; print('All imports OK')"

## 2. Understanding the Chess Environment

Before training, let's understand how we represent chess positions and moves.
We use:
- **FEN** (Forsyth-Edwards Notation) — compact string representation of a board position
- **UCI** (Universal Chess Interface) — move format like `e2e4` (from-square to-square)

In [4]:
import chess

# Create the starting position
board = chess.Board()
print("Starting position FEN:")
print(board.fen())
print()
print(board)

Starting position FEN:
rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R


In [5]:
# Legal moves in UCI format — this is what we'll feed to the model
legal_moves = [m.uci() for m in board.legal_moves]
print(f"{len(legal_moves)} legal moves: {' '.join(legal_moves)}")

20 legal moves: g1h3 g1f3 b1c3 b1a3 h2h3 g2g3 f2f3 e2e3 d2d3 c2c3 b2b3 a2a3 h2h4 g2g4 f2f4 e2e4 d2d4 c2c4 b2b4 a2a4


In [6]:
# Make some moves and see how the board changes
board.push(chess.Move.from_uci("e2e4"))  # King's pawn
board.push(chess.Move.from_uci("e7e5"))  # Symmetric response
board.push(chess.Move.from_uci("g1f3"))  # Knight out
print(board)
print(f"\nFEN: {board.fen()}")
print(f"Legal moves for Black: {' '.join(m.uci() for m in board.legal_moves)}")

r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . p . . .
. . . . P . . .
. . . . . N . .
P P P P . P P P
R N B Q K B . R

FEN: rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2
Legal moves for Black: g8e7 g8h6 g8f6 f8e7 f8d6 f8c5 f8b4 f8a3 e8e7 d8e7 d8f6 d8g5 d8h4 b8c6 b8a6 h7h6 g7g6 f7f6 d7d6 c7c6 b7b6 a7a6 h7h5 g7g5 f7f5 d7d5 c7c5 b7b5 a7a5


In [7]:
# Validate moves — this is how the reward function checks legality
test_board = chess.Board(board.fen())

good_move = chess.Move.from_uci("b8c6")  # Knight to c6 (legal)
bad_move = chess.Move.from_uci("a1a2")   # Rook move (illegal for Black)

print(f"b8c6 is legal: {good_move in test_board.legal_moves}")
print(f"a1a2 is legal: {bad_move in test_board.legal_moves}")

b8c6 is legal: True
a1a2 is legal: False


## 3. Building the Reward Function

The reward function is the **heart of GRPO training**. It scores each model output on a progressive scale:

| Component | Weight | What it checks |
|-----------|--------|----------------|
| `format_reward` | 0.1 | Did the model use `<move>` tags? |
| `legality_reward` | 0.4 | Is the move legal in this position? |
| `quality_reward` | 0.5 | How good is the move (Stockfish eval)? |

This graduated approach is critical — the model starts knowing nothing about chess.
With binary rewards, it would get zero signal. With progressive rewards,
even outputting the right format gets *some* reward, enabling learning.

In [ ]:
import re

def extract_move(text):
    """Extract UCI move from <move>...</move> tags."""
    match = re.search(r"<move>\s*([a-h][1-8][a-h][1-8][qrbn]?)\s*</move>", text)
    return match.group(1) if match else None

# Test with various outputs
test_outputs = [
    "I think the best move is <move>e2e4</move>",        # Good format
    "The move is e2e4",                                  # Missing tags
    "<move>xyz123</move>",                               # Bad UCI format
    "<move>a1a2</move>",                                 # Valid UCI, may be illegal
]

for output in test_outputs:
    move = extract_move(output)
    print(f"  {output[:50]:50s} -> {move}")

  I think the best move is <move>e2e4</move>         -> e2e4
  The move is e2e4                                   -> None
  <move>xyz123</move>                                -> None
  <move>a1a2</move>                                  -> a1a2


In [9]:
# Simulate the full reward pipeline on a position
import math

test_fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
test_board = chess.Board(test_fen)
print(f"Position: {test_fen}")
print(f"Legal moves: {' '.join(m.uci() for m in test_board.legal_moves)}")
print()

test_completions = [
    "<move>e7e5</move>",          # Legal, common response
    "<move>a7a1</move>",          # Valid UCI, but illegal
    "I play e7e5",                # Wrong format
    "<move>d7d5</move>",          # Legal, Scandinavian Defense
]

for comp in test_completions:
    move_str = extract_move(comp)
    fmt = 1.0 if move_str else 0.0
    legal = 0.0
    quality = 0.0
    if move_str:
        try:
            move = chess.Move.from_uci(move_str)
            legal = 1.0 if move in test_board.legal_moves else 0.0
        except ValueError:
            pass
    total = 0.1 * fmt + 0.4 * legal + 0.5 * quality
    print(f"  {comp:30s}  fmt={fmt:.0f}  legal={legal:.0f}  quality={quality:.1f}  total={total:.2f}")

Position: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
Legal moves: g8h6 g8f6 b8c6 b8a6 h7h6 g7g6 f7f6 e7e6 d7d6 c7c6 b7b6 a7a6 h7h5 g7g5 f7f5 e7e5 d7d5 c7c5 b7b5 a7a5

  <move>e7e5</move>               fmt=1  legal=1  quality=0.0  total=0.50
  <move>a7a1</move>               fmt=1  legal=0  quality=0.0  total=0.10
  I play e7e5                     fmt=0  legal=0  quality=0.0  total=0.00
  <move>d7d5</move>               fmt=1  legal=1  quality=0.0  total=0.50


## 4. Preparing Training Data

We use the [Lichess chess puzzles dataset](https://huggingface.co/datasets/Lichess/chess-puzzles) — ~4.5M puzzles with verified best moves from Stockfish analysis.

The data prep script:
1. Loads puzzles rated 1200-2200 (intermediate difficulty)
2. Extracts the puzzle position (after opponent's setup move)
3. Creates **15K examples for SFT** (FEN + best move pairs)
4. Creates **5K examples for GRPO** (FEN positions only — the model discovers good moves via RL)

In [ ]:
# Run data preparation on a free MIG slice (no GPU needed, just CPU + network)
!rv run --mig python scripts/data_prep.py

In [ ]:
# Check the job status
!rv ps

In [ ]:
# View logs
!rv logs

## 5. SFT Warmstart

SFT (Supervised Fine-Tuning) teaches the model **how** to play chess — specifically:
- The prompt format (FEN + legal moves)
- The output format (`<move>...</move>` tags)
- Basic move selection from high-quality games

We use **LoRA** (Low-Rank Adaptation) so we only train ~0.5% of the model's parameters.
This makes training fast and memory-efficient.

**Why SFT first?** RL can amplify existing capabilities but can't teach them from scratch.
Without SFT, the base model makes legal moves ~5% of the time. After SFT, it should reach >90%.

In [ ]:
# Submit SFT training job (1x A100 80GB, ~1-2 hours)
!rv run -g 1 -t a100 --time 2h --name chess-sft python scripts/sft_train.py

In [ ]:
# Monitor the job
!rv ps

In [ ]:
# Follow training logs (Ctrl+C to stop following)
!rv logs chess-sft -f

In [ ]:
# Check GPU utilization
!rv gpu chess-sft

📊 **Check your W&B dashboard** for training metrics — look for the `chess-sft` run.
Key metrics to watch: `train/loss` should decrease steadily.

## 6. GRPO Training

Now the fun part — **reinforcement learning**. GRPO works like this:

1. For each position, generate **8 candidate moves** (the "group")
2. Score each move with our reward function (format + legality + Stockfish quality)
3. Compute **relative advantages** within the group (better moves get positive advantage)
4. Update the model to make high-advantage moves more likely

Unlike PPO, GRPO doesn't need a separate critic network — it uses the group statistics
as a baseline. This makes it simpler and more memory-efficient.

**Architecture:** We use vLLM in **colocate mode** — it shares GPU memory with the trainer
on a single A100 80GB. This avoids needing a second GPU for a separate vLLM server.
The wrapper script `run_grpo.sh` installs vLLM and launches training.

In [ ]:
# Submit GRPO training job (1x A100 80GB, ~20-30 minutes for 500 steps)
!rv run -g 1 -t a100 --time 2h --name chess-grpo bash scripts/run_grpo.sh

In [ ]:
# Monitor
!rv ps

In [ ]:
# Follow GRPO training logs
!rv logs chess-grpo -f

In [ ]:
# Check GPU utilization
!rv gpu chess-grpo

📊 **W&B metrics to watch** (`chess-grpo` run):
- `rewards/format_reward/mean` — should hit 1.0 quickly (model learns the `<move>` tags)
- `rewards/legality_reward/mean` — should climb steadily (model makes fewer illegal moves)
- `rewards/quality_reward/mean` — slower improvement (move quality from Stockfish)
- `reward` — combined weighted signal, should trend upward

## 7. Evaluation — Can We Beat GPT-4o?

Time to test our trained model:
1. **Puzzle accuracy** — legal move rate and Stockfish evaluation on held-out positions
2. **Full games vs GPT-4o** — play actual chess games and track wins/losses

In [ ]:
# Submit evaluation job
!rv run -g 1 -t a100 --time 1h --name chess-eval python scripts/evaluate.py --num_puzzles 100 --num_games 10

In [ ]:
# Follow eval logs
!rv logs chess-eval -f

In [ ]:
# Pull evaluation results (single quotes so $USER expands on the cluster, not locally)
!rv exec 'cat $(ls -t /scratch/$USER/.rv/outputs/chess-eval-*/eval_results.json 2>/dev/null | head -1)'

In [ ]:
import json, subprocess

result = subprocess.run(
    ["rv", "exec", "cat $(ls -t /scratch/$USER/.rv/outputs/chess-eval-*/eval_results.json 2>/dev/null | head -1)"],
    capture_output=True, text=True
)

if result.stdout.strip():
    data = json.loads(result.stdout)
    pr = data["puzzle_results"]
    gr = data["game_results"]
    print(f"Model: {data['model_path']}")
    print(f"\nPuzzle Evaluation ({pr['num_evaluated']} positions):")
    print(f"  Legal move rate: {pr['legal_rate']:.1%}")
    print(f"  Avg Stockfish score: {pr['avg_score']:.1f} cp")
    print(f"\nGames vs GPT-4o ({gr['total']} games):")
    print(f"  Wins: {gr['wins']}  Losses: {gr['losses']}  Draws: {gr['draws']}")
    print(f"  Win rate: {gr['win_rate']:.1%}")
    if gr.get("wins_by_opponent_illegal"):
        print(f"  ({gr['wins_by_opponent_illegal']} wins from opponent illegal moves)")
else:
    print("No eval results found yet — check that the eval job has completed:")
    print("  rv ps -a | grep chess-eval")

## What's Next?

**Results** (SFT 30K examples + GRPO 1000 steps on A100/H100):
- **100% legal move rate** on 100 held-out puzzles — our model never makes an illegal move
- **0W/5L/5D vs GPT-4o** with fair evaluation (10 retries per move, 200-move limit)
- GPT-4o needed **58 total retries** for illegal moves across 10 games — our model needed 0

### Future Research Directions

1. **Game-outcome rewards** — Instead of scoring individual moves with Stockfish, play full games during GRPO and reward wins/draws/losses. This gives the model a signal for multi-move planning and endgame conversion.

2. **Full game transcripts in SFT** — Include complete grandmaster games (from the Lichess game database, not just puzzles) so the model sees opening → middlegame → endgame → checkmate as a coherent arc.

3. **Game history in the prompt** — Pass the move history (or last N moves) alongside the FEN so the model can maintain strategic continuity across moves.

4. **Endgame tablebases** — For positions with ≤7 pieces, there's a mathematically perfect answer. Include these as training data for basic endgame technique (K+R vs K, pawn promotion, etc.).

5. **Smarter reward shaping** — Add a checkmate bonus, material advantage component, or weight endgame positions higher in the quality reward.

6. **Curriculum learning** — Start GRPO with easy endgame positions, then gradually increase difficulty.

7. **Scale up** — Larger base models (14B, 32B), more SFT data (30K → 100K+), more GRPO steps (1000 → 5000+), multi-GPU training.

8. **Better evaluation** — Play against Stockfish at various Elo levels, measure actual Elo rating, evaluate opening/middlegame/endgame separately.

## 8. Play Against Your Model

Now let's play chess against our trained model in the browser! The `web/` directory contains a Next.js app with a drag-and-drop chess board.

**Architecture:**
```
Browser (localhost:3000) → Next.js API route → vLLM on Rivanna A100
```

**Setup (3 terminals):**

1. **Start vLLM server** on Rivanna (serves the model as an OpenAI-compatible API):
```bash
rv run -g 1 -t a100 --time 3h --name chess-serve bash scripts/serve_model.sh
rv logs chess-serve -f  # wait for "Started server process"
```

2. **Forward the port** (tunnels the compute node to localhost):
```bash
rv forward 8000 <JOB_ID>  # use the job ID from rv ps, not the name
```

3. **Start the web app:**
```bash
cd web && pnpm install && pnpm dev
# Open http://localhost:3000
```

> **Note:** `rv forward` requires the numeric job ID (e.g. `rv forward 8000 10202839`), not the job name. Get it from `rv ps`.